# HumanOS — práctica completa + Opción A
Notebook guiado para Google Colab. Ejecuta cada celda en orden. El código principal está en `src/humanos/`.

## 0. Cargar los archivos
Mantén los nombres `responsesdf`, `configdf`, `predictivedf` de tu trabajo. Modifica solo `BASE` si la carpeta cambia.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/My Drive/Colab Notebooks/Silverce'
responsesdf = pd.read_csv(f'{BASE}/responses.csv')
configdf = pd.read_csv(f'{BASE}/item_config.csv')
predictivedf = pd.read_csv(f'{BASE}/predictive_challenge.csv')
print(responsesdf.shape, configdf.shape, predictivedf.shape)

## 1. Obligatoria A: validación y cálculo de señales
Invertimos `6-x` cuando `reverse_flag=True`, transformamos a `25*(x-1)` y calculamos la media ponderada según `weight`.

In [ ]:
import sys
sys.path.insert(0, f'{BASE}/src')
from humanos.core import validate, calculate_signals, organization_signals, moments, cohort_report
validate(responsesdf, configdf)
scored = calculate_signals(responsesdf, configdf)
display(scored.head())
display(pd.DataFrame(organization_signals(scored)))

## 2. Obligatoria B: momentos no exclusivos
Las fórmulas son heurísticas exploratorias y no sustituyen una escala validada. No se usa la edad para inferir un momento.

In [ ]:
momentsdf = moments(scored)
display(momentsdf.head())
display(momentsdf.drop(columns='participant_id').mean().round(2))

## 3. Obligatoria C: cohortes y umbral de privacidad
`n<10` implica `reportable=False`, `score=None`, `delta=None`.

In [ ]:
cohortsdf = pd.DataFrame(cohort_report(scored))
display(cohortsdf.head(12))
display(cohortsdf[~cohortsdf.reportable].head())

## 4. Obligatoria D: API
El servicio está en `src/humanos/api.py`. En tu ordenador ejecuta `uvicorn humanos.api:app --reload` desde la carpeta del proyecto y visita `/docs`.

## 5. Opción A: EDA
El dataset predictivo es independiente del de las señales. La diferencia W2−W1 sirve para descripción, no como feature.

In [ ]:
display(predictivedf.head())
print(predictivedf.info())
print('Nulos:',predictivedf.isna().sum().sum())
print('IDs duplicados:',predictivedf.participant_id.duplicated().sum())
predictivedf['change'] = predictivedf.future_visibility_w2 - predictivedf.future_visibility_w1
display(predictivedf['change'].describe().round(2))

## 6. Visualización del target y cambio

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(predictivedf['change'], bins=25)
ax.axvline(0, linestyle='--', color='black')
ax.set(xlabel='Cambio W2 - W1',ylabel='Participantes',title='Distribución del cambio')
plt.show()

## 7. Modelado reproducible
Target: W2. Se comparan baseline, Ridge, Random Forest y Gradient Boosting. CV se calcula solo sobre train. El preprocesamiento se aprende dentro de cada fold.

In [ ]:
from humanos.predictive import run
report, predictions = run(f'{BASE}/predictive_challenge.csv')
display(pd.DataFrame({name:{**v['test'], 'cv_train_mae':v['cv_train_mae'],'cv_train_r2':v['cv_train_r2']} for name,v in report['model_results'].items()}).T)
display(predictions.head())

## 8. Interpretación y overfitting
Comparar métricas de train, CV y test. Una caída grande de train a test sugiere sobreajuste. Permutation importance expresa pérdida de MAE al permutar una feature; no implica causalidad.

In [ ]:
display(pd.DataFrame(report['model_results']).T)
display(pd.Series(report['ridge_permutation_mae_improvement']).sort_values(ascending=False).to_frame('Incremento MAE al permutar'))

## 9. Conclusiones
El error de test cuantifica precisión en datos reservados. No se infieren causas, diagnósticos personales ni decisiones laborales. Ver README para el análisis del apartado 9 y limitaciones.